In [1]:
import joblib
import pickle
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [2]:
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [3]:
tfidf_fakenews = joblib.load('tfidf_fakenews.pkl')
tfidf_categories = joblib.load('tfidf_categories.pkl')
le = joblib.load('le_categories.pkl')
model_fakenews = pickle.load(open('model_fakenews.pkl','rb'))
model_categories = pickle.load(open('model_categories2.pkl','rb'))
model_fakenews.multi_class = 'auto'
model_categories.multi_class = 'auto'

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

## Функция предсказания

In [5]:
def predict(title, text=''):
    full_text = (title + ' ' + text).strip()
    processed = preprocess(full_text)


    x_fake = tfidf_fakenews.transform([processed])
    fake_label = model_fakenews.predict(x_fake)[0]
    decision = model_fakenews.decision_function(x_fake)[0]
    fk = 1 / (1 + np.exp(-decision))


    x_cat = tfidf_categories.transform([processed])
    cat_label = model_categories.predict(x_cat)[0]
    cat_name = le.inverse_transform([cat_label])[0]

    if fake_label == 1:
        fake_verdict = f"Да, это фейк (наша модерация уверена на {fk})"
    else:
        fake_verdict = f" Нет, новость подтверждена (вероятность фейка равна {fk})"

    print(f"Статья: {full_text}")
    print(f"Фейк: {fake_verdict}")
    print(f"Категория: {cat_name}")
    return fake_label, cat_name



Тест:

In [6]:
print("АМЕРИКАНСКИЙ МАКС")
print("Проверим вашу новость!")
print("-" * 60)
title = input("Заголовок: ")
text  = input("Текст: ")
print("\nИтог")
print("-" * 40)
predict(title=title, text=text)

АМЕРИКАНСКИЙ МАКС
Проверим вашу новость!
------------------------------------------------------------
Заголовок: SHOCK NEWS!
Текст: DONALD TRUMP TOOK GREENLAND BACK TO USA

Итог
----------------------------------------
Статья: SHOCK NEWS! DONALD TRUMP TOOK GREENLAND BACK TO USA
Фейк: Да, это фейк (наша модерация уверена на 0.9802885555477592)
Категория: POLITICS


(np.float64(1.0), 'POLITICS')

In [ ]:
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer

model_fakenews = pickle.load(open('model_fakenews.pkl','rb'))

test_news = [
    """
    The White House announced a new economic plan on Tuesday aimed at supporting small businesses,
    reducing inflationary pressure, and expanding access to affordable loans for local manufacturers.
    According to administration officials, the proposal includes tax credits for companies that invest
    in domestic production, additional funding for infrastructure projects, and measures designed to
    lower supply chain costs. The plan is expected to be reviewed by Congress next month, where lawmakers
    from both parties are likely to debate the size of the spending package and its potential impact on
    the federal budget. Economists said the proposal could provide short-term support for businesses,
    although its long-term effect will depend on implementation and broader market conditions.
    """,

    """
    BREAKING: Secret documents leaked by anonymous insiders reveal that top government officials have
    been hiding a massive plan to control every smartphone in America using invisible signals from new
    wireless towers. According to the report, the technology can allegedly read private messages, track
    thoughts, and silently change people’s opinions without their knowledge. The shocking discovery was
    supposedly confirmed by unnamed experts who claim that former presidents, major technology companies,
    and international organizations are all involved in the operation. Officials have refused to confirm
    the story, which supporters say proves that the truth is being covered up by the media.
    """
]

tfidf = pickle.load("tfidf_fakenews.pkl")

news_tfidf = tfidf.transform(test_news)
preds = model_fakenews.predict(news_tfidf)
probas = model_fakenews.predict_proba(news_tfidf)

prediction_table = pd.DataFrame({
    "text": test_news,
    "fake_probability": probas[:, 1],
})

prediction_table

,text,fake_probability
0,\n The White House announced a new economic...,0.316706
1,\n BREAKING: Secret documents leaked by ano...,0.860777
